In [1]:
pip install opacus

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 MB 39.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 36.6 MB/s eta 0:00:00
  Attempting uninstall: sympy
    Found existing installation: sympy 1.13.2
    Uninstalling sympy-1.13.2:
      Successfully uninstalled sympy-1.13.2
Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install torch torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 36.4 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [172]:
import torch
from torchvision import datasets
from torchvision.datasets import CIFAR10
import torchvision.transforms as transforms
from torchvision.transforms import ToTensor
from torch.utils.data import DataLoader
import torch.nn as nn
from torch import optim
from opacus import PrivacyEngine
from torch.autograd import Variable
import numpy as np

In [218]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465],
                         std=[0.2023, 0.1994, 0.2010])
])

In [219]:
train_data = CIFAR10(
    root = 'data',
    train = True,
    transform = transform,
    download = True,
)
test_data = CIFAR10(
    root = 'data',
    train = False,
    transform = transform
)

In [220]:
loaders = {
    'train' : torch.utils.data.DataLoader(train_data,
                                          batch_size=100,
                                          shuffle=True,),

    'test'  : torch.utils.data.DataLoader(test_data,
                                          batch_size=100,
                                          shuffle=True,),
}

In [270]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 16, 5, 1, 2),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(16, 32, 5, 1, 2),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(32, 64, 5, 1, 2),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.out = nn.Linear(64 * 4 * 4, 10)
    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = x.view(x.size(0), -1)
        output = self.out(x)
        return output, x

In [271]:
cnn = CNN()

In [272]:
cnn

CNN(
  (conv1): Sequential(
    (0): Conv2d(3, 16, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv2): Sequential(
    (0): Conv2d(16, 32, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv3): Sequential(
    (0): Conv2d(32, 64, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (out): Linear(in_features=1024, out_features=10, bias=True)
)

In [273]:
lossf = nn.CrossEntropyLoss()
optimizer = optim.Adam(cnn.parameters(), lr = 0.001)
def acc(predictions,labels):
    return (predictions==labels).mean()

In [274]:
privacy_engine = PrivacyEngine()
cnn, optimizer, loaders['train'] = privacy_engine.make_private_with_epsilon(
    module=cnn,
    optimizer=optimizer,
    data_loader=loaders['train'],
    epochs=25,
    target_epsilon=8,
    target_delta=1e-5,
    max_grad_norm=1.0,
)

In [275]:
num_epochs = 25
def train(num_epochs, cnn, loaders):
    cnn.train()
    total_step = len(loaders['train'])
    for epoch in range(num_epochs):
        overall_acc = []
        overall_loss = []
        for i, (images, labels) in enumerate(loaders['train']):
            b_x = Variable(images)
            b_y = Variable(labels)
            output = cnn(b_x)[0]
            predictions = np.argmax(output.detach().cpu().numpy(), axis=1)
            labels = labels.detach().cpu().numpy()
            accuracy = acc(predictions,labels)
            overall_acc.append(accuracy)
            loss = lossf(output, b_y)
            overall_loss.append(loss.item())
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            if (i+1) % 500 == 0:
                print ('Epoch [{}/{}], Step [{}/{}], Accuracy: {:.4f}, Loss: {:.4f}'.format(epoch + 1, num_epochs, i + 1, total_step, np.mean(overall_acc) * 100, np.mean(overall_loss)))
                pass
            pass
        pass

In [276]:
train(num_epochs, cnn, loaders)

Epoch [1/25], Step [500/500], Accuracy: 30.5278, Loss: 1.9616
Epoch [2/25], Step [500/500], Accuracy: 39.5279, Loss: 1.7980
Epoch [3/25], Step [500/500], Accuracy: 43.0047, Loss: 1.8257
Epoch [4/25], Step [500/500], Accuracy: 45.1972, Loss: 1.8374
Epoch [5/25], Step [500/500], Accuracy: 47.0872, Loss: 1.8721
Epoch [6/25], Step [500/500], Accuracy: 48.4124, Loss: 1.8916
Epoch [7/25], Step [500/500], Accuracy: 49.3451, Loss: 1.9553
Epoch [8/25], Step [500/500], Accuracy: 50.5840, Loss: 1.9437
Epoch [9/25], Step [500/500], Accuracy: 51.3690, Loss: 1.9761
Epoch [10/25], Step [500/500], Accuracy: 52.5677, Loss: 1.9791
Epoch [11/25], Step [500/500], Accuracy: 52.6853, Loss: 1.9995
Epoch [12/25], Step [500/500], Accuracy: 53.0311, Loss: 2.0097
Epoch [13/25], Step [500/500], Accuracy: 54.0083, Loss: 2.0490
Epoch [14/25], Step [500/500], Accuracy: 54.2865, Loss: 2.1146
Epoch [15/25], Step [500/500], Accuracy: 54.8478, Loss: 2.0837
Epoch [16/25], Step [500/500], Accuracy: 55.2576, Loss: 2.0973
E

In [284]:
def test():
    cnn.eval()
    overall_acc = []
    overall_loss = []
    with torch.no_grad():
        for images, labels in loaders['test']:
            test_output, last_layer = cnn(images)
            pred_y = torch.max(test_output, 1)[1].data.squeeze()
            labels = Variable(labels)
            loss = lossf(test_output, labels)
            overall_loss.append(loss.item())
            accuracy = (pred_y == labels).sum().item() / float(labels.size(0))
            labels = labels.detach().cpu().numpy()
            overall_acc.append(accuracy)
            pass
        print('Test Accuracy: %.4f' % (np.mean(overall_acc) * 100) + ', Test Loss: %.4f' % np.mean(overall_loss))
    pass

In [285]:
test()

Test Accuracy: 54.9700, Test Loss: 2.3027
